In [ ]:
from pathlib import Path
from collections import Counter
from typing import Any
import importlib.util
import json
import re

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from tokenizers import Tokenizer
from transformers import AutoConfig, AutoModelForCausalLM, PreTrainedTokenizerFast

In [ ]:
PROJECT_ROOT = Path("..")
MODEL_DIR = PROJECT_ROOT / "model" / "GenNA"
TOKENIZER_JSON_PATH = PROJECT_ROOT / "configs" / "tokenizer.json"

EXCLUDE_SPECIAL_FROM_DENOM = False
EXCLUDE_SPECIAL_FROM_PROJECTION = True

RUN_TSNE = True
TSNE_PREPCA_DIM = 50
TSNE_PERPLEXITY = 30
TSNE_LEARNING_RATE = 200
TSNE_RANDOM_STATE = 42

QUERY_TOKEN = "tggtgaaa"
TOPK = 20
COSINE_BINS = 70

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16 if DEVICE.type == "cuda" else torch.float32
FLASH_ATTN_AVAILABLE = DEVICE.type == "cuda" and importlib.util.find_spec("flash_attn") is not None

for path in [MODEL_DIR, TOKENIZER_JSON_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required path not found: {path}")

print(f"device = {DEVICE}")

In [ ]:
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.size": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "legend.title_fontsize": 10,
    "axes.linewidth": 1.2,
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "axes.unicode_minus": False,
})

CATEGORY_ORDER = [
    "1-mer", "2-mer", "3-mer", "4-mer", "5-mer", "6-mer", "7-mer", "8-mer",
    "9-mer and above", "N-containing kmers", "Non-nucleotide"
]

PURE_ATCG_RE = re.compile(r"^[atcgATCG]+$")
ATCGN_RE = re.compile(r"^[atcgnATCGN]+$")

In [ ]:
def build_tokenizer(tokenizer_json_path: Path) -> PreTrainedTokenizerFast:
    tokenizer_obj = Tokenizer.from_file(str(tokenizer_json_path))
    return PreTrainedTokenizerFast(
        tokenizer_object=tokenizer_obj,
        unk_token="<unk>",
        pad_token="<pad>",
        eos_token="<eos>",
    )


def load_model(model_dir: Path, device: torch.device, dtype: torch.dtype):
    config = AutoConfig.from_pretrained(model_dir)
    model_kwargs = {
        "config": config,
        "torch_dtype": dtype,
        "low_cpu_mem_usage": True,
    }
    if FLASH_ATTN_AVAILABLE:
        model_kwargs["attn_implementation"] = "flash_attention_2"
    model = AutoModelForCausalLM.from_pretrained(model_dir, **model_kwargs).to(device)
    model.eval()
    return model


def get_id_to_token(tokenizer_fast: PreTrainedTokenizerFast) -> dict[int, str]:
    vocab = tokenizer_fast.get_vocab()
    id_to_token = {idx: tok for tok, idx in vocab.items()}
    return dict(sorted(id_to_token.items(), key=lambda item: item[0]))


def normalize_token_for_check(token: str) -> str:
    if token.endswith("</w>"):
        token = token[:-4]
    return token.replace("\u200b", "").replace("\u200c", "").replace("\u200d", "")


def is_special_token(token: str) -> bool:
    return (token.startswith("<") and token.endswith(">")) or (token.startswith("[") and token.endswith("]"))


def classify_token_and_len(token: str) -> tuple[str, int]:
    norm = normalize_token_for_check(token)
    if not norm:
        return "non", 0
    if PURE_ATCG_RE.fullmatch(norm):
        return "pure", len(norm)
    if ATCGN_RE.fullmatch(norm) and "n" in norm.lower():
        return "withN", len(norm)
    return "non", 0


def bucket_name_by_len(length: int) -> str:
    if 1 <= length <= 8:
        return f"{length}-mer"
    if length >= 9:
        return "9-mer and above"
    return "Non-nucleotide"


def build_vocab_metadata(id_to_token: dict[int, str]) -> pd.DataFrame:
    rows = []
    for token_id, raw_token in id_to_token.items():
        kind, length = classify_token_and_len(raw_token)
        if kind == "pure":
            category = bucket_name_by_len(length)
        elif kind == "withN":
            category = "N-containing kmers"
        else:
            category = "Non-nucleotide"

        rows.append({
            "token_id": token_id,
            "raw_token": raw_token,
            "normalized_token": normalize_token_for_check(raw_token),
            "is_special": is_special_token(raw_token),
            "kind": kind,
            "nucleotide_length": length if kind in {"pure", "withN"} else 0,
            "category": category,
        })

    return pd.DataFrame(rows).sort_values("token_id").reset_index(drop=True)


def cosine_similarity_matrix(x: np.ndarray) -> np.ndarray:
    x = x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-12, None)
    return x @ x.T


def compute_category_centroids(emb_matrix: np.ndarray, df_meta: pd.DataFrame, category_order: list[str]):
    centroids = []
    labels = []
    rows = []

    for category in category_order:
        mask = (df_meta["category"] == category).values
        if mask.sum() == 0:
            continue
        center = emb_matrix[mask].mean(axis=0)
        centroids.append(center)
        labels.append(category)
        rows.append({
            "category": category,
            "count": int(mask.sum()),
            "centroid_norm": float(np.linalg.norm(center)),
        })

    return np.stack(centroids, axis=0), labels, pd.DataFrame(rows)

In [ ]:
tokenizer = build_tokenizer(TOKENIZER_JSON_PATH)
id_to_token = get_id_to_token(tokenizer)
df_meta = build_vocab_metadata(id_to_token)

denom_df = df_meta[~df_meta["is_special"]] if EXCLUDE_SPECIAL_FROM_DENOM else df_meta
summary = (
    denom_df["category"]
    .value_counts()
    .reindex(CATEGORY_ORDER, fill_value=0)
    .rename_axis("Category")
    .reset_index(name="Count")
)
summary["Proportion"] = summary["Count"] / summary["Count"].sum()

print(f"Total vocabulary size: {len(df_meta)}")
print(f"Denominator size: {len(denom_df)}")
display(summary)

In [ ]:
def plot_vocab_donut(summary_df: pd.DataFrame) -> None:
    labels = summary_df["Category"].tolist()
    sizes = summary_df["Count"].tolist()
    total = sum(sizes)

    fig, ax = plt.subplots(figsize=(12, 7), dpi=150)
    colors = plt.cm.tab20(np.linspace(0.05, 0.95, len(sizes)))
    cutoff = 0.04

    def autopct(pct):
        return "" if pct / 100 < cutoff else f"{pct:.1f}%"

    ax.pie(
        sizes,
        autopct=autopct,
        startangle=90,
        pctdistance=0.75,
        colors=colors,
        wedgeprops={"width": 0.45, "edgecolor": "white", "linewidth": 2},
        textprops={"color": "black", "fontsize": 15, "fontweight": "bold"},
    )
    ax.text(0, 0, f"Vocabulary\n{total}", ha="center", va="center", fontsize=22, fontweight="bold")

    col_dot_x, col_label_x, col_num_x = 1.05, 1.12, 1.7
    y_start, header_gap, line_spacing = 0.88, 0.06, 0.07
    ax.text(col_label_x, y_start, "Token Type", transform=ax.transAxes, fontsize=17, fontweight="bold", va="bottom", ha="left")
    ax.text(col_num_x, y_start, "Count", transform=ax.transAxes, fontsize=17, fontweight="bold", va="bottom", ha="right")

    for i, (label, size, color) in enumerate(zip(labels, sizes, colors)):
        y_pos = y_start - header_gap - i * line_spacing
        ax.scatter(col_dot_x, y_pos, marker="o", s=120, color=color, transform=ax.transAxes, clip_on=False)
        ax.text(col_label_x, y_pos, label, transform=ax.transAxes, fontsize=16, va="center", ha="left")
        ax.text(col_num_x, y_pos, f"{size}", transform=ax.transAxes, fontsize=16, va="center", ha="right")

    plt.subplots_adjust(left=0.05, right=0.6, top=0.9, bottom=0.1)
    plt.show()


plot_vocab_donut(summary)

In [ ]:
def plot_kmer_coverage(summary_df: pd.DataFrame) -> None:
    from matplotlib.ticker import PercentFormatter

    count_map = dict(zip(summary_df["Category"], summary_df["Count"]))
    k_vals = list(range(1, 9))
    pure_counts = [count_map.get(f"{k}-mer", 0) for k in k_vals]
    possible = [4 ** k for k in k_vals]
    coverage = [count / total if total > 0 else 0.0 for count, total in zip(pure_counts, possible)]

    fig, ax = plt.subplots(figsize=(11, 6.5), dpi=150)
    x = np.arange(len(k_vals))
    colors = plt.cm.tab20(np.linspace(0.05, 0.95, 11))[:8]
    bars = ax.bar(x, coverage, color=colors, edgecolor="black", linewidth=1.2, width=0.75, zorder=3)

    for rect, count, total, ratio in zip(bars, pure_counts, possible, coverage):
        height = rect.get_height()
        ax.text(rect.get_x() + rect.get_width() / 2, height + 0.02, f"{ratio:.1%}", ha="center", va="bottom", fontsize=18)
        ax.text(rect.get_x() + rect.get_width() / 2, height + 0.095, f"{count}/{total}", ha="center", va="bottom", fontsize=16, color="#666666")

    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    ax.set_ylim(0, 1.25)
    ax.set_ylabel("Coverage (Count / $4^k$)", fontsize=22, labelpad=10)
    ax.set_xlabel("k-mer length", fontsize=22, labelpad=10)
    ax.set_xticks(x)
    ax.set_xticklabels([str(k) for k in k_vals], fontsize=20)
    ax.tick_params(axis="y", labelsize=18)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()


plot_kmer_coverage(summary)

In [ ]:
model = load_model(MODEL_DIR, DEVICE, DTYPE)
embedding_weight = model.get_input_embeddings().weight.detach().float().cpu().numpy()

if embedding_weight.shape[0] != len(df_meta):
    print(f"[Warning] embedding rows = {embedding_weight.shape[0]}, vocab size = {len(df_meta)}")
    max_n = min(embedding_weight.shape[0], len(df_meta))
    embedding_weight = embedding_weight[:max_n]
    df_meta = df_meta.iloc[:max_n].copy().reset_index(drop=True)

print("Embedding matrix shape:", embedding_weight.shape)

In [ ]:
def plot_category_heatmap(sim_df: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(4.5, 4), dpi=300)
    im = ax.imshow(sim_df.values, vmin=-1, vmax=1, cmap="viridis")
    ax.set_xticks(np.arange(sim_df.shape[1]))
    ax.set_yticks(np.arange(sim_df.shape[0]))
    ax.set_xticklabels(sim_df.columns, rotation=45, ha="right")
    ax.set_yticklabels(sim_df.index)

    for i in range(sim_df.shape[0]):
        for j in range(sim_df.shape[1]):
            ax.text(j, i, f"{sim_df.iloc[i, j]:.2f}", ha="center", va="center", fontsize=6.5, color="black")

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Centroid cosine similarity", fontsize=10)
    plt.tight_layout()
    plt.show()


centroids, centroid_labels, centroid_norms_df = compute_category_centroids(embedding_weight, df_meta, CATEGORY_ORDER)
centroid_sim = cosine_similarity_matrix(centroids)
centroid_sim_df = pd.DataFrame(centroid_sim, index=centroid_labels, columns=centroid_labels)

display(centroid_norms_df)
plot_category_heatmap(centroid_sim_df)

In [ ]:
if EXCLUDE_SPECIAL_FROM_PROJECTION:
    projection_mask = ~df_meta["is_special"].values
    df_proj = df_meta.loc[projection_mask].copy().reset_index(drop=True)
    X_proj = embedding_weight[projection_mask]
else:
    df_proj = df_meta.copy().reset_index(drop=True)
    X_proj = embedding_weight.copy()

pca_3d = PCA(n_components=3, random_state=42)
X_pca_3d = pca_3d.fit_transform(X_proj)

df_pca_3d = df_proj.copy()
df_pca_3d["pca1"] = X_pca_3d[:, 0]
df_pca_3d["pca2"] = X_pca_3d[:, 1]
df_pca_3d["pca3"] = X_pca_3d[:, 2]

print("Explained variance ratio:", pca_3d.explained_variance_ratio_)
print("Total variance explained:", float(pca_3d.explained_variance_ratio_.sum()))

In [ ]:
def plot_projection_3d(df_plot: pd.DataFrame, x_col: str, y_col: str, z_col: str) -> None:
    fig = plt.figure(figsize=(6, 5), dpi=300)
    ax = fig.add_subplot(111, projection="3d")
    cmap = plt.cm.tab20
    color_map = {cat: cmap(i / max(1, len(CATEGORY_ORDER) - 1)) for i, cat in enumerate(CATEGORY_ORDER)}

    for category in CATEGORY_ORDER:
        sub = df_plot[df_plot["category"] == category]
        if sub.empty:
            continue
        is_non_nuc = category == "Non-nucleotide"
        ax.scatter(
            sub[x_col], sub[y_col], sub[z_col],
            s=1.5 if not is_non_nuc else 0.8,
            alpha=0.85 if not is_non_nuc else 0.35,
            color=color_map[category], edgecolors="none", label=category,
        )

    ax.set_xlabel(x_col.upper(), fontsize=8, labelpad=5)
    ax.set_ylabel(y_col.upper(), fontsize=8, labelpad=5)
    ax.set_zlabel(z_col.upper(), fontsize=8, labelpad=5)
    ax.tick_params(axis="both", which="major", labelsize=6)
    ax.view_init(elev=20, azim=45)
    leg = ax.legend(title="Category", bbox_to_anchor=(1.1, 1), loc="upper left", frameon=False)
    handles = leg.legend_handles if hasattr(leg, "legend_handles") else leg.legendHandles
    for handle in handles:
        handle.set_sizes([15.0])
        handle.set_alpha(1.0)
    plt.tight_layout()
    plt.show()


plot_projection_3d(df_pca_3d, "pca1", "pca2", "pca3")

In [ ]:
def plot_projection_2d(df_plot: pd.DataFrame, x_col: str, y_col: str) -> None:
    fig, ax = plt.subplots(figsize=(5.5, 4), dpi=300)
    cmap = plt.cm.tab20
    color_map = {cat: cmap(i / max(1, len(CATEGORY_ORDER) - 1)) for i, cat in enumerate(CATEGORY_ORDER)}

    for category in CATEGORY_ORDER:
        sub = df_plot[df_plot["category"] == category]
        if sub.empty:
            continue
        is_non_nuc = category == "Non-nucleotide"
        ax.scatter(
            sub[x_col], sub[y_col],
            s=1.5 if not is_non_nuc else 0.8,
            alpha=0.75 if not is_non_nuc else 0.35,
            color=color_map[category], edgecolors="none", label=category,
        )

    ax.set_xlabel(x_col.upper(), fontsize=9)
    ax.set_ylabel(y_col.upper(), fontsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    leg = ax.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    handles = leg.legend_handles if hasattr(leg, "legend_handles") else leg.legendHandles
    for handle in handles:
        handle.set_sizes([15.0])
        handle.set_alpha(1.0)
    plt.tight_layout()
    plt.show()


if RUN_TSNE:
    prepca_dim = min(TSNE_PREPCA_DIM, X_proj.shape[1], max(2, X_proj.shape[0] - 1))
    if prepca_dim >= 2 and prepca_dim < X_proj.shape[1]:
        pca_pre = PCA(n_components=prepca_dim, random_state=TSNE_RANDOM_STATE)
        X_tsne_input = pca_pre.fit_transform(X_proj)
    else:
        X_tsne_input = X_proj

    tsne = TSNE(
        n_components=2,
        perplexity=TSNE_PERPLEXITY,
        learning_rate=TSNE_LEARNING_RATE,
        init="pca",
        random_state=TSNE_RANDOM_STATE,
    )
    X_tsne = tsne.fit_transform(X_tsne_input)

    df_tsne = df_proj.copy()
    df_tsne["tsne1"] = X_tsne[:, 0]
    df_tsne["tsne2"] = X_tsne[:, 1]
    plot_projection_2d(df_tsne, "tsne1", "tsne2")

In [ ]:
def plot_token_cosine_distribution(token: str, id_to_token: dict[int, str], emb_norm: torch.Tensor, vocab: dict[str, int], topk: int = 20, bins: int = 70) -> pd.DataFrame:
    if token not in vocab:
        raise KeyError(f"Token not found in vocab: {token}")

    token_id = vocab[token]
    query_vec = emb_norm[token_id]
    sims = emb_norm @ query_vec
    sims_np = sims.numpy()

    mask = np.ones_like(sims_np, dtype=bool)
    mask[token_id] = False
    sims_bg = sims_np[mask]

    mean_sim = float(np.mean(sims_bg))
    std_sim = float(np.std(sims_bg))
    p99 = float(np.quantile(sims_bg, 0.99))

    vals, idxs = torch.topk(sims, k=topk + 1)
    rows = []
    for idx, val in zip(idxs.tolist(), vals.tolist()):
        if idx == token_id:
            continue
        rows.append({
            "rank": len(rows) + 1,
            "token_id": idx,
            "token": id_to_token[idx],
            "cosine_similarity": float(val),
        })
        if len(rows) >= topk:
            break

    df_topk = pd.DataFrame(rows)
    top1 = float(df_topk.iloc[0]["cosine_similarity"])

    x_left = min(np.quantile(sims_bg, 0.001), mean_sim - 4 * std_sim)
    x_right_main = np.quantile(sims_bg, 0.9995)
    x_right = max(x_right_main, top1 * 1.08)

    fig, ax = plt.subplots(figsize=(4.5, 3.5), dpi=300)
    counts, _, _ = ax.hist(sims_bg, bins=bins, range=(x_left, x_right_main), color="#4C78A8", alpha=0.85, edgecolor="white", linewidth=0.3)
    ymax = counts.max() if len(counts) > 0 else 1

    ax.axvline(mean_sim, linestyle="--", linewidth=1.0, color="#7F7F7F")
    ax.axvline(p99, linestyle=(0, (4, 2)), linewidth=1.0, color="#E17C05")
    ax.axvline(top1, linestyle="-", linewidth=1.2, color="#C44E52")

    x_offset = (x_right - x_left) * 0.015
    ax.text(mean_sim - x_offset, ymax * 1.03, f"mean\n{mean_sim:.3f}", ha="right", va="bottom", fontsize=9, color="#555555")
    ax.text(p99 - x_offset, ymax * 1.03, f"p99\n{p99:.3f}", ha="right", va="bottom", fontsize=9, color="#A85A00")
    ax.text(top1 - x_offset, ymax * 1.03, f"top-1\n{top1:.3f}", ha="right", va="bottom", fontsize=9, color="#9E2F33")

    ax.set_xlabel("Cosine similarity", fontsize=10)
    ax.set_ylabel("Count", fontsize=10)
    ax.set_xlim(x_left, x_right)
    ax.set_ylim(0, ymax * 1.18)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return df_topk


vocab = tokenizer.get_vocab()
emb_tensor = torch.tensor(embedding_weight)
emb_norm_tensor = emb_tensor / emb_tensor.norm(dim=1, keepdim=True).clamp(min=1e-12)

df_topk = plot_token_cosine_distribution(
    token=QUERY_TOKEN,
    id_to_token=id_to_token,
    emb_norm=emb_norm_tensor,
    vocab=vocab,
    topk=TOPK,
    bins=COSINE_BINS,
)
display(df_topk)